In [1]:
pip install pandas numpy openpyxl xlsxwriter html5lib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import os
from io import BytesIO
import warnings
warnings.filterwarnings("ignore", "This pattern is interpreted as a regular expression")

# ================================
# CONFIGURATION
# ================================
SIGNED_HTML_PATH = r'C:\Users\Administrator\Documents\GitHub\FM24-Filter_JG\File Templates\Signings.html' #Transfers
LOANS_HTML_PATH = r'C:\Users\Administrator\Documents\GitHub\FM24-Filter_JG\File Templates\Loans.html' #Loans
UNIVERSAL_HTML_PATH = r'C:\Users\Administrator\Documents\GitHub\FM24-Filter_JG\File Templates\Universe.html'
OUTPUT_PATH = r'C:\Users\Administrator\Documents\GitHub\FM24-Filter_JG\File Templates\ATM.xlsx'
LEAGUE_MULTIPLIERS_PATH = r'C:\Users\Administrator\Documents\GitHub\FM24-Filter_JG\File Templates\League Multipliers.xlsx'  # Path to your Excel file

TEXT_COLUMNS = [
    'UID', 'Name', 'Rec', 'EU National', 'Position', 'Pros',
    'Preferred Foot', 'Inf', 'Transfer Value', 'Nat', 'Division', 'Club', 'Personality'
]

PERCENTAGE_COLUMNS = ['Sv %', 'OP-Cr %', 'Hdr %', 'Conv %', 'Pas %', 'Cr C/A', 'Tck R', 'Pens Saved Ratio', 'Pen/R', 'Shot %']

# ================================
# LOAD LEAGUE POWER RATINGS FROM EXCEL
# ================================
def load_league_power(file_path):
    try:
        # Read the Excel file
        df = pd.read_excel(file_path)
        
        # Convert to dictionary with 'League' as key and 'Power Rating' as value
        league_power = dict(zip(df['League'], df['Power Rating']))
        
        # Add default 'Others' value if not present in the file
        if 'Others' not in league_power:
            league_power['Others'] = 5
            
        return league_power
    except Exception as e:
        print(f"Error loading league power ratings: {e}")
        # Return a default dictionary if file can't be loaded
        return {'Others': 5}

# Load the league power ratings
LEAGUE_POWER = load_league_power(LEAGUE_MULTIPLIERS_PATH)

# ================================
# LEAGUE NAME FIXES FOR ENCODING ISSUES
# ================================
LEAGUE_NAME_FIXES = {
    'BrasileirÃ£o AssaÃ­ SÃ©rie A': 'Brasileirão Assaí Série A',
    'Primera FederaciÃ³n Grupo I': 'Primera Federación Grupo I',
    'Liga Profesional de FÃºtbol': 'Liga Profesional de Fútbol',
    'Primera FederaciÃ³n Grupo I': 'Primera Federación Grupo',
    'Primera FederaciÃ³n Grupo III': 'Primera Federación Grupo',
    'Primera FederaciÃ³n Grupo IV': 'Primera Federación Grupo',
    'Primera FederaciÃ³n Grupo V': 'Primera Federación Grupo',
    'Primera FederaciÃ³n Grupo VI': 'Primera Federación Grupo',
    'Primera FederaciÃ³n Grupo VII': 'Primera Federación Grupo',
    'Regionalliga SÃ¼dwest': 'Regionalliga Südwest',
    'Serie C NOW Girone A': 'Serie C NOW',
    'Serie C NOW Girone B': 'Serie C NOW',
    'Serie C NOW Girone C': 'Serie C NOW',
    'Spor Toto SÃ¼per Lig': 'Spor Toto Süper Lig',
    'French National 3 - Group A': 'French National 3',
    'French National 3 - Group B': 'French National 3',
    'French National 3 - Group C': 'French National 3',
    'French National 3 - Group D': 'French National 3',
    'French National 3 - Group E': 'French National 3',
    'French National 3 - Group F': 'French National 3',
    'French National 3 - Group G': 'French National 3',
    'French National 3 - Group H': 'French National 3',
    'French National 3 - Group I': 'French National 3',
    'French National 3 - Group J': 'French National 3',
    'French National 3 - Group K': 'French National 3',
    'French National 3 - Group L': 'French National 3',
    'BrasileirÃ£o Serie B Chevrolet': 'Brasileirão Serie B Chevrolet',
    'Serie D Girone A': 'Serie D',
    'Serie D Girone B': 'Serie D',
    'Serie D Girone C': 'Serie D',
    'Serie D Girone D': 'Serie D',
    'Serie D Girone E': 'Serie D',
    'Serie D Girone F': 'Serie D',
    'Serie D Girone G': 'Serie D',
    'Serie D Girone H': 'Serie D',
    'Serie D Girone I': 'Serie D',
    'Serie D Girone J': 'Serie D',
    'Serie D Girone K': 'Serie D',
    'Regionalliga West': 'Regionalliga',
    'Regionalliga Nord': 'Regionalliga',
    'Regionalliga Südwest': 'Regionalliga',
    'Regionalliga Bayern': 'Regionalliga',
    'Regionalliga Nordost': 'Regionalliga',
    'Russian Second Division A Gold': 'Russian Second Division A',
    'Russian Second Division A Silver': 'Russian Second Division A',  
    'Russian Second Division A Bronze': 'Russian Second Division A',    
    'Russian Second Division B - Group 1': 'Russian Second Division B',
    'Russian Second Division B - Group 2': 'Russian Second Division B',
    'Russian Second Division B - Group 3': 'Russian Second Division B',
    'DR Congo Premier Division A': 'DR Congolese Premier Division',
    'DR Congo Premier Division B': 'DR Congolese Premier Division',

    # Add more fixes as needed
}

def fix_league_names(df):
    """Fix encoding issues in league names in the DataFrame"""
    if 'Division' in df.columns:
        df['Division'] = df['Division'].replace(LEAGUE_NAME_FIXES)
    return df

# ================================
# LOAD HTML FILES
# ================================
if not os.path.exists(SIGNED_HTML_PATH):
    raise FileNotFoundError(f"Signed players HTML not found: {SIGNED_HTML_PATH}")

if not os.path.exists(UNIVERSAL_HTML_PATH):
    raise FileNotFoundError(f"Universal players HTML not found: {UNIVERSAL_HTML_PATH}")

if not os.path.exists(LOANS_HTML_PATH):
    raise FileNotFoundError(f"Loan players HTML not found: {LOANS_HTML_PATH}")

with open(SIGNED_HTML_PATH, 'r', encoding='utf-8') as file:
    signed_tables = pd.read_html(file)
df_signed = signed_tables[0].copy()
df_signed = fix_league_names(df_signed)

with open(UNIVERSAL_HTML_PATH, 'r', encoding='utf-8') as file:
    universal_tables = pd.read_html(file)
df_universal = universal_tables[0].copy()
df_universal = fix_league_names(df_universal)

with open(LOANS_HTML_PATH, 'r', encoding='utf-8') as file:
    loans_tables = pd.read_html(file)
df_loans = loans_tables[0].copy()
df_loans = fix_league_names(df_loans)

#runtime 22secs

In [4]:
# After loading each HTML file, before concatenating them
def fix_column_alignment(df):
    """Fix column alignment issues in the dataframe"""
    
    # The actual column order from your HTML (based on your sample)
    expected_columns = [
        'Club', 'Yel', 'xG', 'Tall', 'Tgls', 'Starts', 'Shutouts', 'Red', 'Pts/Gm', 'PoM',
        'Pens S', 'Pens Saved', 'Pens Faced', 'Pens', 'NP-xG', 'Last Gl', 'Last C', 'Mins/Gm',
        'Last 5 Games', 'Last 5 FT Games', 'Int Conc', 'Int Av Rat', 'Int Ast', 'Int Apps',
        'Conc', 'Gls', 'Won', 'G. Mis', 'Lost', 'D', 'Gwin', 'Cln/90', 'Av Rat', 'Mins/Gl',
        'Ast', 'Apps', 'AT Lge Gls', 'AT Lge Apps', 'AT Gls', 'Tck C', 'Tck A', 'Shot %',
        'ShT', 'Shts Blckd', 'Shots', 'Svt', 'Svp', 'Svh', 'Sv %', 'Pr Passes', 'Pres C',
        'Pres A', 'Ps C', 'Pas A', 'OP-KP', 'OP-Crs C', 'OP-Crs A', 'Off', 'K Tck', 'K Pas',
        'Itc', 'Hdrs', 'Goals Outside Box', 'FK Shots', 'xSv %', 'xGP', 'xG/shot', 'Drb',
        'Dist/90', 'Distance', 'Cr C', 'Cr A', 'Cr C/A', 'Conv %', 'Clr/90', 'Clear', 'CCC',
        'Ch C/90', 'Blk/90', 'Blk', 'Inf', 'Name', 'Club', 'Position', 'Age', 'Transfer Value',
        'Rec', 'Aer A/90', 'xA', 'Asts/90', 'UID', 'Saves/90', 'Tgls/90', 'Tcon/90', 'Tck/90',
        'Tck R', 'Shot/90', 'ShT/90', 'Shots Outside Box/90', 'Shts Blckd/90', 'Pr passes/90',
        'Pres C/90', 'Pres A/90', 'Poss Won/90', 'Poss Lost/90', 'Pen/R', 'Pens Saved Ratio',
        'Ps C/90', 'Pas %', 'Ps A/90', 'OP-KP/90', 'OP-Crs C/90', 'OP-Cr %', 'OP-Crs A/90',
        'NP-xG/90', 'Gl Mst', 'Mins', 'K Tck/90', 'K Ps/90', 'K Hdrs/90', 'Int/90', 'Sprints/90',
        'Hdr %', 'Hdrs W/90', 'Hdrs L/90', 'All/90', 'xGP/90', 'xG-OP', 'xG/90', 'xA/90',
        'Drb/90', 'FA', 'Waive Comp for Mgr Role', 'Fls', 'Gls/90', 'Cr C/90', 'Crs A/90',
        'G. Con', 'Position', 'Expires', 'EU National', 'Hdrs A', 'AT Apps', 'Division'
    ]
    
    # If we have more columns than expected, something is wrong
    if len(df.columns) > len(expected_columns):
        print(f"Warning: DataFrame has {len(df.columns)} columns, expected {len(expected_columns)}")
        # Try to identify the position column by looking at column names
        position_cols = [col for col in df.columns if 'Position' in col]
        print(f"Position columns found: {position_cols}")
        
        # If we have two position columns, the second one likely has the data
        if len(position_cols) >= 2:
            pos_col_with_data = position_cols[1]  # The second Position column
            print(f"Using {pos_col_with_data} as the source for position data")
            df['Position'] = df[pos_col_with_data]
    
    return df

# Apply this fix to each dataframe right after loading
df_signed = fix_column_alignment(df_signed)
df_universal = fix_column_alignment(df_universal)
df_loans = fix_column_alignment(df_loans)

Position columns found: ['Position', 'Position.1']
Using Position.1 as the source for position data
Position columns found: ['Position', 'Position.1']
Using Position.1 as the source for position data
Position columns found: ['Position', 'Position.1']
Using Position.1 as the source for position data


In [5]:
# After loading all dataframes, inspect the first row to find position data
print("\n=== INSPECTING FIRST ROW OF LOANS DATA ===")
first_row = df_loans.iloc[0]
for col in df_loans.columns:
    value = first_row[col]
    if pd.notna(value) and str(value).strip() and 'AM' in str(value) or 'D ' in str(value) or 'ST' in str(value):
        print(f"Column '{col}' contains: '{value}'")


=== INSPECTING FIRST ROW OF LOANS DATA ===
Column 'Position' contains: 'AM (RL), ST (C)'
Column 'Position.1' contains: 'AM (RL), ST (C)'


In [6]:

# ================================
# UID‑BASED MERGE AND LABEL  ✅ Fixed version
# ================================

def normalize_uid(df):
    df = df.copy()
    # Convert to float → Int64 (nullable integer) → string
    df['UID'] = pd.to_numeric(df['UID'], errors='coerce')  # force numeric, NaNs stay if bad
    df['UID'] = df['UID'].astype('Int64')                  # keep NaNs clean
    df['UID'] = df['UID'].astype(str).str.strip()          # final string
    return df

# 1️⃣ Normalize UIDs
df_signed    = normalize_uid(df_signed)
df_universal = normalize_uid(df_universal)
df_loans = normalize_uid(df_loans)

# 2️⃣ Add the signability flags
df_signed['Signability']    = 'Available for Transfer'
df_universal['Signability'] = 'Not Transferrable'
df_loans['Signability'] = 'Available on Loan'


# 3️⃣ Concatenate signables first, drop duplicate UIDs

df = (
    pd.concat([df_signed, df_loans, df_universal], ignore_index=True)
      .drop_duplicates(subset='UID', keep='first')
      .reset_index(drop=True)
)

# ================================
# ADD THIS RIGHT HERE - FIX DUPLICATE COLUMN ISSUE
# ================================
# The actual position data is in Position.1, not Position
if 'Position.1' in df.columns:
    print(f"Found Position.1 column with data. Using it as the main Position column.")
    # Replace the empty Position column with Position.1
    df['Position'] = df['Position.1']
    # Drop the duplicate column
    df = df.drop(columns=['Position.1'])
    
# Also check for other potential position columns (like Position.2 if exists)
position_cols = [col for col in df.columns if 'Position' in col]
if len(position_cols) > 1:
    print(f"Warning: Still have multiple position columns: {position_cols}")
    # If there's another Position column, check if it has data
    for col in position_cols[1:]:  # Skip the first one (which should now be our main Position)
        if df[col].notna().any():
            print(f"Column {col} has data. Using it to fill missing positions.")
            df['Position'] = df['Position'].fillna(df[col])
        df = df.drop(columns=[col])

print(f"Final position columns: {[col for col in df.columns if 'Position' in col]}")


# ================================
# FIX DUPLICATE NAME COLUMN ISSUE
# ================================
# Check for duplicate Name columns
name_cols = [col for col in df.columns if 'Name' in col]
print(f"Name columns found: {name_cols}")

if len(name_cols) > 1:
    print(f"Found multiple Name columns. Consolidating...")
    # Find which Name column actually has data
    for col in name_cols:
        non_null_count = df[col].notna().sum()
        print(f"  Column '{col}' has {non_null_count} non-null values")
    
    # Use the column with most data as the main Name
    main_name_col = max(name_cols, key=lambda col: df[col].notna().sum())
    print(f"Using '{main_name_col}' as the main Name column")
    
    # Replace the original Name column
    df['Name'] = df[main_name_col]
    
    # Drop all other Name columns except the original
    for col in name_cols:
        if col != 'Name':
            df = df.drop(columns=[col])

print(f"Final Name column: {df['Name'].notna().sum()} non-null values")
print(f"Sample names: {df['Name'].dropna().head(10).tolist()}")

# ================================
# CLEANING & CONVERSION
# ================================
def clean_and_convert_data(df):
    def convert_percentage_to_float(df, column):
        if column in df.columns:
            df[column] = (df[column].astype(str)
                          .str.replace('%', '')
                          .replace('-', np.nan)
                          .astype(float))
        return df

    # Convert minutes
    if 'Mins' in df.columns:
        df['Mins'] = pd.to_numeric(df['Mins'], errors='coerce')
        df = df[df['Mins'] >= 900].copy()

    # Convert percentage stats
    for col in PERCENTAGE_COLUMNS:
        df = convert_percentage_to_float(df, col)

    # Handle 'Dist/90' - e.g., "7.3mi"
    if 'Dist/90' in df.columns:
        df['Dist/90'] = (
            df['Dist/90']
            .astype(str)
            .str.extract(r'([\d.]+)')[0]
        )

        # Drop only if the string was completely missing
        df['Dist/90'] = pd.to_numeric(df['Dist/90'], errors='coerce')

    print("Dist/90 (after cleaning):")
    print(df['Dist/90'].describe())
    print(df['Dist/90'].dropna().head(10))






    

    # Convert all other numerical columns (except text & signability)
    for col in df.columns:
        if col not in TEXT_COLUMNS + ['Signability']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Handle 'Dist/90' - e.g., "7.3mi"
    if 'Dist/90' in df.columns:
        df['Dist/90'] = (
            df['Dist/90']
            .astype(str)
            .str.extract(r'([\d.]+)')[0]
        )

        # Drop only if the string was completely missing
        df['Dist/90'] = pd.to_numeric(df['Dist/90'], errors='coerce')

    print("Dist/90 (after cleaning):")
    print(df['Dist/90'].describe())
    print(df['Dist/90'].dropna().head(10))


    # Add league strength multiplier
    def get_league_multiplier(division):
        return LEAGUE_POWER.get(division, LEAGUE_POWER['Others']) / 100.0

    df['League Multiplier'] = df['Division'].apply(get_league_multiplier)
    return df

# ✅ Apply cleaning
df = clean_and_convert_data(df)

# ================================
# CONVERT RAW STATS TO PER 90
# ================================
RAW_STATS_TO_PER90 = {
    "Yel": "Yellow/90",
    "Red": "Red/90",
    "Fls": "FoulsMade/90",
    "FA": "FoulsAgainst/90",
    "Off": "Offsides/90",
    "Gl Mst": "Gl Mst/90",
    "Goals Outside Box": "Goals Outside Box/90",
    "FK Shots": "FKShots/90"
}

for raw_stat, per90_stat in RAW_STATS_TO_PER90.items():
    if raw_stat in df.columns:
        df[raw_stat] = pd.to_numeric(df[raw_stat], errors='coerce')
        df['Mins'] = pd.to_numeric(df['Mins'], errors='coerce')
        df[per90_stat] = (df[raw_stat] / (df['Mins'] / 90)).fillna(0)
    else:
        print(f"Warning: Raw stat '{raw_stat}' not found in DataFrame")

print(df['Dist/90'].dropna().unique()[:50])
print(df['Dist/90'].dtype)

#runtime 1sec

Found Position.1 column with data. Using it as the main Position column.
Final position columns: ['Position']
Name columns found: ['Name']
Final Name column: 7743 non-null values
Sample names: ['- -', '- -', '- -', '- -', '- -', '- -', '- -', '- -', '- -', '- -']
Dist/90 (after cleaning):
count    7743.000000
mean        3.524667
std         3.305329
min         0.000000
25%         0.000000
50%         3.200000
75%         7.100000
max         9.000000
Name: Dist/90, dtype: float64
0    7.1
1    7.4
2    8.1
3    3.9
4    6.8
5    8.6
6    8.0
7    7.0
8    2.9
9    8.8
Name: Dist/90, dtype: float64
Dist/90 (after cleaning):
count    7743.000000
mean        3.524667
std         3.305329
min         0.000000
25%         0.000000
50%         3.200000
75%         7.100000
max         9.000000
Name: Dist/90, dtype: float64
0    7.1
1    7.4
2    8.1
3    3.9
4    6.8
5    8.6
6    8.0
7    7.0
8    2.9
9    8.8
Name: Dist/90, dtype: float64
[7.1 7.4 8.1 3.9 6.8 8.6 8.  7.  2.9 8.8 3.7 5.3

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11780\3048207351.py:154: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['League Multiplier'] = df['Division'].apply(get_league_multiplier)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11780\3048207351.py:178: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[per90_stat] = (df[raw_stat] / (df['Mins'] / 90)).fillna(0)


In [7]:
df["Expires"] = df["Expires"].astype("string")
print(df["Expires"].dtype)

string


In [8]:
# Show overall info
print("==== DataFrame Info ====")
print(df.info())

print("\n==== Column Summary ====")
for col in df.columns:
    print(f"\n📌 Column: {col}")
    print(f"Type: {df[col].dtype}")
    print(f"Missing: {df[col].isna().sum()} / {len(df)}")
    
    if df[col].dtype == 'object':
        unique_vals = df[col].dropna().unique()
        print(f"Unique Values: {len(unique_vals)}")
        print("Sample:", unique_vals[:10])
    
    elif pd.api.types.is_numeric_dtype(df[col]):
        print(df[col].describe())

    else:
        print("Sample:", df[col].dropna().head(5).tolist())

==== DataFrame Info ====
<class 'pandas.DataFrame'>
Index: 7743 entries, 0 to 7743
Columns: 157 entries, Pref. to FKShots/90
dtypes: float64(143), str(13), string(1)
memory usage: 9.3 MB
None

==== Column Summary ====

📌 Column: Pref.
Type: float64
Missing: 7189 / 7743
count    554.000000
mean      13.259928
std       14.822797
min        1.000000
25%        6.000000
50%       10.000000
75%       14.000000
max       99.000000
Name: Pref., dtype: float64

📌 Column: Preferred Foot
Type: str
Missing: 0 / 7743
Sample: ['Right', 'Right', 'Right', 'Right', 'Left']

📌 Column: Club
Type: str
Missing: 0 / 7743
Sample: ['Al-Hilal (KSA)', 'Al-Hilal (KSA)', 'Al-Ahli (KSA)', 'A. Madrid', 'Al-Hilal (KSA)']

📌 Column: Yel
Type: float64
Missing: 0 / 7743
count    7743.000000
mean        1.382668
std         2.075467
min         0.000000
25%         0.000000
50%         0.000000
75%         2.000000
max        16.000000
Name: Yel, dtype: float64

📌 Column: xG
Type: float64
Missing: 0 / 7743
count    77

In [9]:
if df['Dist/90'].isna().any():
    print("⚠️ Warning: Missing or zero values found in Dist/90")

In [10]:

"""
# ================================
# DATA PREPROCESSING
# ================================
if 'Mins' in df.columns:
    df['Mins'] = pd.to_numeric(df['Mins'], errors='coerce')
    print(f"Minutes stats before filtering:")
    print(df['Mins'].describe())
    df = df[df['Mins'] >= 900].copy()
    print(f"After minutes filter: {len(df)} players remain")
"""

# ================================
# CREATE PER-90 COLUMNS IF THEY DON'T EXIST
# ================================
# List of raw stats that need per-90 versions
raw_to_per90 = {
    'Sprints': 'Sprints/90',
    'Int': 'Int/90',
    'K Tck': 'K Tck/90',
    'xA': 'xA/90',
    'Ch C': 'Ch C/90',
    'Gls': 'Gls/90',
    'NP-xG': 'NP-xG/90',
    'xGP': 'xGP/90',
    'Hdrs W': 'Hdrs W/90',
    'Tck': 'Tck/90',
    'OP-Crs C': 'OP-Crs C/90',
    'Drb': 'Drb/90',
    'Off': 'Offsides/90',
    'Poss Won': 'Poss Won/90',
    'Poss Lost': 'Poss Lost/90',
    'FoulsMade': 'FoulsMade/90',  # Note: you have 'Fls' not 'FoulsMade'
}

# Create missing per-90 columns
for raw_base, per90_col in raw_to_per90.items():
    if per90_col not in df.columns:
        # Try to find the raw column
        raw_col = None
        if raw_base in df.columns:
            raw_col = raw_base
        elif raw_base == 'FoulsMade' and 'Fls' in df.columns:
            raw_col = 'Fls'
        elif raw_base == 'Off' and 'Off' in df.columns:
            raw_col = 'Off'
        
        if raw_col:
            df[per90_col] = df[raw_col] / (df['Mins'] / 90)
            df[per90_col] = df[per90_col].fillna(0)
            print(f"Created {per90_col} from {raw_col}")
        else:
            df[per90_col] = 0
            print(f"⚠️ Created placeholder {per90_col} (raw column not found)")

# ================================
# CREATE COMPOSITE METRICS (WITH SAFE COLUMN ACCESS)
# ================================
# Intensity
if 'Sprints/90' in df.columns and 'Dist/90' in df.columns:
    df['Intensity'] = df["Sprints/90"] / df["Dist/90"].replace(0, 1)
else:
    df['Intensity'] = 0
    print("⚠️ Created placeholder Intensity")

# Poss_Quality
if 'Poss Won/90' in df.columns and 'Poss Lost/90' in df.columns:
    df['Poss_Quality'] = df["Poss Won/90"] / (df["Poss Lost/90"] + 0.1)
else:
    df['Poss_Quality'] = 0
    print("⚠️ Created placeholder Poss_Quality")

# Proactivity_Index
if all(col in df.columns for col in ['Int/90', 'K Tck/90', 'FoulsMade/90']):
    df['Proactivity_Index'] = (df['Int/90'] + df['K Tck/90']) / (df['FoulsMade/90'] + 1)
else:
    df['Proactivity_Index'] = 0
    print("⚠️ Created placeholder Proactivity_Index")

# Creativity_Index
if all(col in df.columns for col in ['xA/90', 'Ch C/90']):
    df['Creativity_Index'] = (df['xA/90'] * 0.7) + (df['Ch C/90'] * 0.3)
else:
    df['Creativity_Index'] = 0
    print("⚠️ Created placeholder Creativity_Index")

# Finishing_Index
if all(col in df.columns for col in ['Gls/90', 'NP-xG/90']):
    df['Finishing_Index'] = df['Gls/90'] / (df['NP-xG/90'] + 0.01)
else:
    df['Finishing_Index'] = 0
    print("⚠️ Created placeholder Finishing_Index")

print(f"\n✅ After creating metrics: {len(df)} players")

# ================================
# CHECK REQUIRED COLUMNS FOR COMPOSITE METRICS
# ================================
required_columns = ['Sprints/90', 'Dist/90', 'Poss Won/90', 'Poss Lost/90', 
                    'Int/90', 'K Tck/90', 'FoulsMade/90', 'xA/90', 'Ch C/90', 
                    'Gls/90', 'NP-xG/90']

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    print(f"⚠️ Warning: Missing columns for composite metrics: {missing_columns}")
    print("Creating placeholder columns with zeros to avoid errors...")
    for col in missing_columns:
        df[col] = 0

# Define numeric columns to scale
TEXT_COLUMNS = ['EU National', 'Name', 'UID', 'Position', 'Personality', 'Preferred Foot', 
                'Rec', 'Club', 'Transfer Value', 'Division', 'Nat', 'Inf', 'Signability','Expires']
                
numeric_columns = [col for col in df.columns 
                   if col not in TEXT_COLUMNS 
                   and pd.api.types.is_numeric_dtype(df[col])]

# Scale numeric columns (0-1)
for col in numeric_columns:
    if col == 'Age': continue 
    if df[col].nunique() > 1:
        min_val, max_val = df[col].min(), df[col].max()
        df[col] = (df[col] - min_val) / (max_val - min_val)
    else:
        df[col] = 0.5
        
        
# First, ensure Position column is string type
if 'Position' in df.columns:
    df['Position'] = df['Position'].astype(str)

# ================================
# UPDATED ARCHETYPE FORMULAS - FIXED FOR ACTUAL POSITION FORMAT
# ================================
ARCHETYPE_FORMULAS = {
    "Sweeper Keeper": {
        "filter": df['Position'].str.contains("GK", case=False, na=False),
        "formula": lambda d: (
            0.70 * (d.get("xGP/90", 0) * 0.7 + d.get("Sv %", 0) * 0.3) +   # Vital Two
            0.20 * d.get("Pas %", 0) +                                     # System Fit
            0.10 * (1 - d.get("Gl Mst/90", 0))                             # Negative Metric Check
        ),
        "label": "SK Rating"
    },
    
    "Central Defender": {
        "filter": df['Position'].str.contains(r"D\s*\(C\)", regex=True, na=False),
        "formula": lambda d: (
            0.70 * (d.get("Hdrs W/90", 0) * 0.6 + d.get("Tck/90", 0) * 0.4) + 
            0.20 * d.get("Proactivity_Index", 0) + 
            0.10 * (1 - d.get("Gl Mst/90", 0))
        ),
        "label": "CD Rating"
    },
    
    "Complete Wingback": {
        "filter": df['Position'].str.contains(r"(D|WB)\s*\([RL]+\)", regex=True, na=False),
        "formula": lambda d: (
            0.70 * (d.get("OP-Crs C/90", 0) * 0.6 + d.get("Intensity", 0) * 0.4) + 
            0.20 * d.get("xA/90", 0) + 
            0.10 * d.get("Poss_Quality", 0)
        ),
        "label": "CWB Rating"
    },
    
    "Defensive Midfielder": {
        "filter": df['Position'].str.contains(r"DM", regex=True, na=False),
        "formula": lambda d: (
            0.70 * (d.get("Int/90", 0) * 0.6 + d.get("Ps %", 0) * 0.4) + 
            0.20 * d.get("Proactivity_Index", 0) + 
            0.10 * d.get("Tck R", 0)
        ),
        "label": "DM Rating"
    },
    
    "Inside Forward": {
        "filter": df['Position'].str.contains(r"AM\s*\([RL]+\)", regex=True, na=False),
        "formula": lambda d: (
            0.70 * (d.get("NP-xG/90", 0) * 0.5 + d.get("Drb/90", 0) * 0.5) + 
            0.20 * d.get("Intensity", 0) + 
            0.10 * d.get("Creativity_Index", 0)
        ),
        "label": "IF Rating"
    },
    
    "Attacking Midfielder": {
        "filter": df['Position'].str.contains(r"AM\s*\(C\)", regex=True, na=False),
        "formula": lambda d: (
            0.70 * (d.get("xA/90", 0) * 0.6 + d.get("Ch C/90", 0) * 0.4) + 
            0.20 * d.get("Pr passes/90", 0) + 
            0.10 * d.get("Pas %", 0)
        ),
        "label": "AM Rating"
    },
    
    "Striker": {
        "filter": df['Position'].str.contains(r"ST", regex=True, na=False),
        "formula": lambda d: (
            0.70 * (d.get("NP-xG/90", 0) * 0.6 + d.get("Intensity", 0) * 0.4) + 
            0.20 * d.get("xG/shot", 0) + 
            0.10 * (1 - d.get("Offsides/90", 0))
        ),
        "label": "ST Rating"
    }
}

# ================================
# DEBUG: Check if any data will be written
# ================================
print(f"\n📊 DataFrame shape: {df.shape}")
print(f"Columns available: {list(df.columns)}")
print(f"Sample of first 5 rows:")
print(df.head(5)[['UID', 'Name', 'Position'] if all(col in df.columns for col in ['UID', 'Name', 'Position']) else df.columns[:5]])

# Check if any archetype filters will produce results
for role, config in ARCHETYPE_FORMULAS.items():
    if 'filter' in config:
        filtered_count = config['filter'].sum() if hasattr(config['filter'], 'sum') else 0
        print(f"  {role}: {filtered_count} players match filter")

# ================================
# OUTPUT TO EXCEL WITH SUMMARY SHEET
# ================================
output_excel = BytesIO()

with pd.ExcelWriter(output_excel, engine='xlsxwriter', engine_kwargs={'options': {'strings_to_urls': False}}) as writer:
    workbook = writer.book
    unicode_format = workbook.add_format({'font_name': 'Arial Unicode MS', 'valign': 'vcenter'})
    
    # First create the summary sheet at the beginning
    worksheet = workbook.add_worksheet('Player Archetype Summary')
    writer.sheets['Player Archetype Summary'] = worksheet
    
    # Dictionary to store archetype data for summary
    summary_data = []
    
    # Process all archetypes
    for role, config in ARCHETYPE_FORMULAS.items():
        role_df = df[config["filter"]].copy()
        if role_df.empty:
            continue

        # Calculate ratings
        role_df[config["label"]] = config["formula"](role_df)
        adjusted_label = f"Adjusted {config['label']}"
        role_df[adjusted_label] = role_df[config["label"]] * role_df['League Multiplier']
        
        # Calculate percentile for each player within this archetype
        role_df['Percentile'] = role_df[adjusted_label].rank(pct=True)
        
        # Store data for summary
        top_players = role_df[role_df['Percentile'] > 0.95][['UID', 'Name', 'Percentile']].copy()
        top_players['Archetype'] = role
        summary_data.append(top_players)
        
        # Write archetype sheet
        role_df['Ranking'] = role_df[adjusted_label].rank(method='min', ascending=False)
        
        output_cols = [
            'UID', 'Name', 'Age', 'Personality', 'Signability', 'EU National', 'Position', 'Preferred Foot',
            'Transfer Value', 'Nat', 'Division', 'Club',
            config["label"], adjusted_label, 'Percentile', 'Ranking', 'League Multiplier', 'Expires'
        ]
        
        result_df = role_df[output_cols].copy().sort_values(by=adjusted_label, ascending=False)
        result_df.to_excel(writer, sheet_name=role, index=False)
        
        # Format archetype sheet
        archetype_sheet = writer.sheets[role]
        for i, col in enumerate(output_cols):
            archetype_sheet.set_column(i, i, 20 if col != 'Name' else 30, unicode_format)
        
        # Format percentile as percentage
        if 'Percentile' in output_cols:
            percent_format = workbook.add_format({'num_format': '0.0%'})
            percentile_col_idx = output_cols.index('Percentile')
            archetype_sheet.set_column(percentile_col_idx, percentile_col_idx, 12, percent_format)
    
    # ================================
    # BUILD SUMMARY SHEET DATA
    # ================================
    if summary_data:
        all_archetypes = pd.concat(summary_data).sort_values(['UID', 'Percentile'], ascending=[True, False])
        
        # Group and format archetypes
        def format_archetypes(group):
            return ", ".join(f"{row['Archetype']} ({row['Percentile']:.1%})" for _, row in group.iterrows())
        
        player_summary = (
            all_archetypes.groupby(['UID', 'Name'])
            .apply(format_archetypes)
            .reset_index(name='Top Archetypes (>95%)')
        )
        
        # Add additional info
        player_summary = player_summary.merge(
            df[['UID', 'Position', 'Age', 'Nat', 'Club', 'Division', 'Personality', 'Signability', 'Transfer Value']],
            on='UID', how='left'
        )
        
        # Final columns and sorting
        player_summary = player_summary[[
            'UID', 'Name', 'Position', 'Club', 'Division', 
            'Signability', 'Transfer Value', 'Age', 'Nat', 'Personality', 'Top Archetypes (>95%)'
        ]]
        player_summary['Archetype Count'] = player_summary['Top Archetypes (>95%)'].str.count(',') + 1
        player_summary = player_summary.sort_values(['Archetype Count', 'Name'], ascending=[False, True])
        
        # Write to summary sheet
        player_summary.to_excel(writer, sheet_name='Player Archetype Summary', index=False)
        
        # Format summary sheet
        summary_sheet = writer.sheets['Player Archetype Summary']
        summary_sheet.set_column('A:A', 15)  # UID
        summary_sheet.set_column('B:B', 30)  # Name
        summary_sheet.set_column('C:C', 15)  # Position
        summary_sheet.set_column('D:D', 25)  # Club
        summary_sheet.set_column('E:E', 25)  # Division
        summary_sheet.set_column('F:F', 15)  # Signability
        summary_sheet.set_column('G:G', 15)  # Transfer Value
        summary_sheet.set_column('H:H', 60)  # Age
        summary_sheet.set_column('I:I', 60)  # Nationality
        summary_sheet.set_column('J:J', 60)  # Personality
        summary_sheet.set_column('K:K', 60)  # Top Archetypes

# ================================
# SAVE FILE
# ================================
try:
    with open(OUTPUT_PATH, 'wb') as output_file:
        output_file.write(output_excel.getbuffer())
    print(f"\n✅ Success! Output file saved:\n{OUTPUT_PATH}")
    print(f"Includes {len(ARCHETYPE_FORMULAS)} archetype sheets + summary sheet")
except Exception as e:
    print(f"\n❌ Error saving file: {e}")
    print("Please check directory permissions or disk space.")


✅ After creating metrics: 7743 players

📊 DataFrame shape: (7743, 162)
Columns available: ['Pref.', 'Preferred Foot', 'Club', 'Yel', 'xG', 'Tall', 'Tgls', 'Starts', 'Shutouts', 'Red', 'Pts/Gm', 'PoM', 'Pens S', 'Pens Saved', 'Pens Faced', 'Pens', 'NP-xG', 'Last Gl', 'Last C', 'Mins/Gm', 'Last 5 Games', 'Last 5 FT Games', 'Int Conc', 'Int Av Rat', 'Int Ast', 'Int Apps', 'Conc', 'Gls', 'Won', 'G. Mis', 'Lost', 'D', 'Gwin', 'Cln/90', 'Av Rat', 'Mins/Gl', 'Ast', 'Apps', 'AT Lge Gls', 'AT Lge Apps', 'AT Gls', 'Tck C', 'Tck A', 'Shot %', 'ShT', 'Shts Blckd', 'Shots', 'Svt', 'Svp', 'Svh', 'Sv %', 'Pr Passes', 'Pres C', 'Pres A', 'Ps C', 'Pas A', 'OP-KP', 'OP-Crs C', 'OP-Crs A', 'Off', 'K Tck', 'K Pas', 'Itc', 'Hdrs', 'Goals Outside Box', 'FK Shots', 'xSv %', 'xGP', 'xG/shot', 'Drb', 'Dist/90', 'Distance', 'Cr C', 'Cr A', 'Cr C/A', 'Conv %', 'Clr/90', 'Clear', 'CCC', 'Ch C/90', 'Blk/90', 'Blk', 'Inf', 'Name', 'Club.1', 'Position', 'Age', 'Transfer Value', 'Rec', 'Aer A/90', 'xA', 'Asts/90', '

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11780\3414611970.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Intensity'] = df["Sprints/90"] / df["Dist/90"].replace(0, 1)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11780\3414611970.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Poss_Quality'] = df["Poss Won/90"] / (df["Poss Lost/90"] + 0.1)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11780\3414611970.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result o


✅ Success! Output file saved:
C:\Users\Administrator\Documents\GitHub\FM24-Filter_JG\File Templates\ATM.xlsx
Includes 7 archetype sheets + summary sheet


In [11]:
# Test each archetype with dummy perfect player
test_player = {col: 1 for col in numeric_columns}
for name, archetype in ARCHETYPE_FORMULAS.items():
    print(f"{name}: {archetype['formula'](test_player)}")

Sweeper Keeper: 0.8999999999999999
Central Defender: 0.8999999999999999
Complete Wingback: 0.9999999999999999
Defensive Midfielder: 0.72
Inside Forward: 0.9999999999999999
Attacking Midfielder: 0.9999999999999999
Striker: 0.8999999999999999


In [12]:
# ================================
# COMPREHENSIVE COLUMN INSPECTION
# ================================
print("\n" + "="*80)
print("COLUMN INSPECTION")
print("="*80)

# Get first row of data
first_row = df.iloc[0]

# Print all columns with their first value
print("\n📋 ALL COLUMNS WITH SAMPLE VALUES (first row):")
print("-" * 80)
for col in df.columns:
    value = first_row[col]
    value_str = str(value) if pd.notna(value) else "NaN"
    # Truncate long values
    if len(value_str) > 50:
        value_str = value_str[:50] + "..."
    print(f"{col:30} : {value_str}")

# Find columns that might contain text/names
print("\n" + "="*80)
print("🔍 SEARCHING FOR COLUMNS WITH TEXT DATA")
print("="*80)

text_columns = []
for col in df.columns:
    if df[col].dtype == 'object':  # Text columns
        # Get sample of non-null values
        sample = df[col].dropna().head(20).tolist()
        if sample:
            # Filter out numeric-looking strings and placeholders
            text_values = [str(v) for v in sample 
                          if isinstance(v, str) 
                          and len(str(v)) > 1 
                          and not str(v).replace('.', '').replace('-', '').strip().isdigit()
                          and str(v) not in ['-', '--', '- -', 'NaN', 'nan', '']]
            
            if text_values:
                unique_text = list(set(text_values))[:5]
                print(f"\n📌 Column: '{col}'")
                print(f"   Total non-null: {df[col].notna().sum()}")
                print(f"   Sample values: {unique_text}")
                text_columns.append(col)

# Check specifically for player names
print("\n" + "="*80)
print("👤 LOOKING FOR PLAYER NAMES")
print("="*80)

name_patterns = ['Name', 'PLAYER', 'Player', 'NAME', 'name', 'Nom', 'Prenom', 'Surname']
for pattern in name_patterns:
    matching_cols = [col for col in df.columns if pattern in col]
    if matching_cols:
        for col in matching_cols:
            sample = df[col].dropna().head(10).tolist()
            print(f"\nColumn '{col}' (matches '{pattern}'):")
            print(f"  Sample: {sample}")

# Show value counts for potential text columns
print("\n" + "="*80)
print("📊 VALUE COUNTS FOR TEXT COLUMNS")
print("="*80)

for col in text_columns[:10]:  # Limit to first 10 text columns
    print(f"\n{col} (top 10 values):")
    print(df[col].value_counts().head(10))

# Check if UID can be used as identifier
print("\n" + "="*80)
print("🔑 UID COLUMN INFO")
print("="*80)
if 'UID' in df.columns:
    print(f"UID - unique count: {df['UID'].nunique()}")
    print(f"UID - sample: {df['UID'].dropna().head(10).tolist()}")
else:
    print("No UID column found!")

print("\n" + "="*80)
print("INSPECTION COMPLETE")
print("="*80)


COLUMN INSPECTION

📋 ALL COLUMNS WITH SAMPLE VALUES (first row):
--------------------------------------------------------------------------------
Pref.                          : NaN
Preferred Foot                 : Right
Club                           : Al-Hilal (KSA)
Yel                            : 0.3125
xG                             : 0.08377983438869946
Tall                           : NaN
Tgls                           : 0.6625
Starts                         : 0.4444444444444444
Shutouts                       : NaN
Red                            : 0.2857142857142857
Pts/Gm                         : 0.87
PoM                            : 0.18181818181818182
Pens S                         : 0.0
Pens Saved                     : NaN
Pens Faced                     : NaN
Pens                           : 0.0
NP-xG                          : 0.08730964467005076
Last Gl                        : NaN
Last C                         : NaN
Mins/Gm                        : 0.8395696161875701


In [14]:
# ================================
# LIST ALL UNUSED COLUMNS
# ================================
print("\n" + "="*80)
print("📊 UNUSED COLUMNS ANALYSIS")
print("="*80)

# Get all columns in the dataframe
all_columns = set(df.columns.tolist())
print(f"Total columns in dataframe: {len(all_columns)}")

# Define all columns that ARE used in your code
used_columns = set()

# 1. Columns used in TEXT_COLUMNS
used_columns.update(TEXT_COLUMNS)
used_columns.update(['Display_Name'])  # If you added this

# 2. Columns used in PERCENTAGE_COLUMNS
used_columns.update(PERCENTAGE_COLUMNS)

# 3. Columns used in RAW_STATS_TO_PER90 (both raw and per90)
for raw_stat, per90_stat in RAW_STATS_TO_PER90.items():
    used_columns.add(raw_stat)
    used_columns.add(per90_stat)

# 4. Columns used in raw_to_per90 mapping
for raw_base, per90_col in raw_to_per90.items():
    used_columns.add(per90_col)
    # Add possible raw column names
    if raw_base in df.columns:
        used_columns.add(raw_base)
    elif raw_base == 'FoulsMade' and 'Fls' in df.columns:
        used_columns.add('Fls')
    elif raw_base == 'Off' and 'Off' in df.columns:
        used_columns.add('Off')

# 5. Columns used in composite metrics
composite_metrics = ['Intensity', 'Poss_Quality', 'Proactivity_Index', 
                     'Creativity_Index', 'Finishing_Index']
used_columns.update(composite_metrics)

# 6. Columns used in archetype formulas
archetype_columns = [
    'xGP/90', 'Sv %', 'Pas %', 'Gl Mst/90', 'Hdrs W/90', 'Tck/90', 
    'OP-Crs C/90', 'xA/90', 'Int/90', 'Ps %', 'Tck R', 'NP-xG/90', 
    'Drb/90', 'Ch C/90', 'Pr passes/90', 'xG/shot', 'Offsides/90',
    'League Multiplier', 'Mins', 'Dist/90', 'Sprints/90', 'Poss Won/90',
    'Poss Lost/90', 'K Tck/90', 'FoulsMade/90', 'Gls/90'
]
used_columns.update(archetype_columns)

# 7. Columns used in filtering and output
filter_columns = ['Position', 'Mins', 'Age']
used_columns.update(filter_columns)

# 8. Columns that are essential for the output structure
essential_columns = ['UID', 'Name', 'Club', 'Division', 'Signability', 
                     'EU National', 'Preferred Foot', 'Transfer Value', 
                     'Nat', 'Personality', 'Expires', 'Rec', 'Inf']
used_columns.update(essential_columns)

# 9. League Multiplier column
used_columns.add('League Multiplier')

# 10. Per-90 columns created from raw stats
per90_created = ['Yellow/90', 'Red/90', 'FoulsMade/90', 'FoulsAgainst/90', 
                 'Offsides/90', 'Gl Mst/90', 'Goals Outside Box/90', 'FKShots/90']
used_columns.update(per90_created)

# Filter to only include columns that actually exist in the dataframe
used_columns = {col for col in used_columns if col in df.columns}

# Find unused columns
unused_columns = all_columns - used_columns

print(f"\n✅ Used columns: {len(used_columns)}")
print(f"❌ Unused columns: {len(unused_columns)}")

# Show unused columns grouped by category
print("\n" + "="*80)
print("📋 UNUSED COLUMNS BY CATEGORY")
print("="*80)

# Categorize unused columns
def categorize_column(col):
    col_lower = col.lower()
    if 'per90' in col_lower or '/90' in col_lower or 'per 90' in col_lower:
        return "Per-90 Stats"
    elif any(x in col_lower for x in ['tck', 'tackl', 'def', 'header', 'hdr', 'blk', 'clr', 'int']):
        return "Defensive Stats"
    elif any(x in col_lower for x in ['shot', 'goal', 'gls', 'xG', 'np-xg', 'finish']):
        return "Attacking Stats"
    elif any(x in col_lower for x in ['pass', 'ps ', 'pr ', 'kp', 'cross', 'crs', 'ch c', 'xa']):
        return "Passing/Creativity"
    elif any(x in col_lower for x in ['sprint', 'dist', 'intensity', 'pace']):
        return "Physical Stats"
    elif any(x in col_lower for x in ['yel', 'red', 'fls', 'fa', 'off', 'foul']):
        return "Discipline Stats"
    elif any(x in col_lower for x in ['gk', 'sv', 'save', 'xgp', 'pens saved']):
        return "Goalkeeper Stats"
    elif any(x in col_lower for x in ['age', 'mins', 'apps', 'starts']):
        return "Playing Time"
    elif any(x in col_lower for x in ['club', 'div', 'league', 'nation', 'nat']):
        return "Context Info"
    else:
        return "Other"

# Group unused columns by category
unused_by_category = {}
for col in sorted(unused_columns):
    category = categorize_column(col)
    if category not in unused_by_category:
        unused_by_category[category] = []
    unused_by_category[category].append(col)

# Print unused columns by category
for category, cols in unused_by_category.items():
    print(f"\n📌 {category} ({len(cols)} columns):")
    # Print in rows of 5 for readability
    for i in range(0, len(cols), 5):
        row_cols = cols[i:i+5]
        print(f"   {', '.join(row_cols)}")

# Summary statistics
print("\n" + "="*80)
print("📊 SUMMARY STATISTICS")
print("="*80)
print(f"Total columns: {len(all_columns)}")
print(f"Columns used in analysis: {len(used_columns)}")
print(f"Columns not used: {len(unused_columns)}")
print(f"Usage rate: {len(used_columns)/len(all_columns)*100:.1f}%")

# Optional: Show sample of used columns
print("\n✅ SAMPLE OF USED COLUMNS (first 20):")
used_list = sorted(list(used_columns))[:20]
for i, col in enumerate(used_list):
    print(f"  {i+1:2}. {col}")

print("\n" + "="*80)


📊 UNUSED COLUMNS ANALYSIS
Total columns in dataframe: 162

✅ Used columns: 72
❌ Unused columns: 90

📋 UNUSED COLUMNS BY CATEGORY

📌 Playing Time (6 columns):
   AT Apps, AT Lge Apps, Apps, Mins/Gl, Mins/Gm
   Starts

📌 Attacking Stats (4 columns):
   AT Gls, AT Lge Gls, Shots, Tgls

📌 Per-90 Stats (25 columns):
   Aer A/90, All/90, Asts/90, Blk/90, Cln/90
   Clr/90, Cr C/90, Crs A/90, Hdrs L/90, K Hdrs/90
   K Ps/90, OP-Crs A/90, OP-KP/90, Pres A/90, Pres C/90
   Ps A/90, Ps C/90, Saves/90, ShT/90, Shot/90
   Shots Outside Box/90, Shts Blckd/90, Tcon/90, Tgls/90, xG/90

📌 Other (34 columns):
   Ast, Av Rat, CCC, Clear, Conc
   Cr A, Cr C, D, G. Con, G. Mis
   Gwin, Itc, K Pas, Last 5 FT Games, Last 5 Games
   Last C, Last Gl, Lost, Pas A, Pens
   Pens S, PoM, Pref., Pres A, Pres C
   Pts/Gm, ShT, Shts Blckd, Shutouts, Tall
   Waive Comp for Mgr Role, Won, xG, xG-OP

📌 Defensive Stats (9 columns):
   Blk, Hdrs, Hdrs A, Int Apps, Int Ast
   Int Av Rat, Int Conc, Tck A, Tck C

📌 Context 